# Spin-1 XY numerical evidence for Sec. 6

This notebook generates the numerical evidence used by the spin-1 $XY$ example in the draft.  It follows the main-text order:

1. verify the exact staggered tower and the three local witnesses $A_R$, $Z_R$, and $Y_R$;
2. construct the symmetry-resolved microcanonical ensemble for $H_{XY}+H_3$;
3. generate the ETH-scatter and thermal-convergence panels;
4. test preserving and non-preserving deformations, including finite and spatially varying $D_r$.

The default sizes are deliberately modest.  Set `RUN_L12=True` or enlarge `SIZES` on the remote server for production data.

## Imports and run controls

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import (
    LocalWitnessTemplate,
    adjacent_gap_ratio_report,
    basis_permutation_from_variable_permutation,
    cage_compatibility_hierarchy_from_hamiltonians,
    cage_jacobian_conditioning_from_hamiltonian,
    cyclic_symmetry_sector_basis,
    diagnose_cage_stability,
    diagnose_eigenpair,
    diagnose_local_channel_spectrum,
    directed_transition_witness_template,
    eigenstate_expectations,
    gaussian_spectral_filter,
    evaluate_local_witness_on_diagonal_ensemble,
    evaluate_local_witness_on_states,
    hermitianize_local_witness_template,
    linearized_cage_obstruction,
    permutation_matrix,
    project_operator_to_sector,
    project_state_to_sector,
    refine_sector_by_involution,
    select_microcanonical_window_by_count,
    select_microcanonical_window_by_width,
    spectral_observable_moments,
    thermal_activity_margin_from_samples,
    thermodynamic_energy_window_plan,
)
from qlinks.models import (
    SpinOneXYChainModel,
    spin_one_xy_periodic_range_couplings,
    spin_one_xy_phase_compatibility,
    spin_one_xy_scar_tower_states,
    spin_one_xy_tower_thermal_activities,
)
from helpers import save_prx_figure, set_revtex_matplotlib_style

DATA_DIR = ROOT / "experimental" / "data" / "spin1_xy_draft_evidence"
FIGURE_DIR = DATA_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

TOL = 1.0e-10
RUN_L12 = False
USE_TEX = False  # set True for final manuscript rendering with a full TeX installation
SIZES = (8, 10, 12) if RUN_L12 else (8, 10)
TOTAL_SZ = -2
J_DRAFT = 1.0
J1_MATRIX = 2.0 * J_DRAFT  # qlinks matrix element; draft bond action is 2J
J3_OVER_J = 0.10
J3_MATRIX = 2.0 * J3_OVER_J * J_DRAFT
D_THERMAL = 0.63
WINDOW_PREFACTORS = (0.75, 1.0, 1.25)
PRIMARY_WINDOW_PREFACTOR = 1.0
SMOOTH_SIGMA_PREFACTOR = 1.0

print("data directory:", DATA_DIR)
print("sizes:", SIZES, "fixed total Sz:", TOTAL_SZ)
print("draft J:", J_DRAFT, "qlinks nearest-neighbor matrix element:", J1_MATRIX)

set_revtex_matplotlib_style(base_font_size=12, prefer_tex=USE_TEX)


FIGURE_FORMATS = ("pdf", "svg")

def save_spin_figure(fig, stem: str, *, aliases=()):
    save_prx_figure(fig, stem, directory=FIGURE_DIR, formats=FIGURE_FORMATS)
    for alias in aliases:
        if alias != stem:
            save_prx_figure(fig, alias, directory=FIGURE_DIR, formats=FIGURE_FORMATS)



## Model, local witnesses, and symmetry-sector helpers

In [ ]:
def make_spin1_witnesses(*, xy_matrix_element: float = J1_MATRIX):
    # Y_r=(Sz_r)^2-1 is represented on its only nonzero local pattern |0>.
    y_template = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=((0,),),
        local_operator=np.asarray([[-1.0]], dtype=np.complex128),
        metadata={"name": "Y_r", "support_sites": 1, "channel_type": "diagonal"},
    )

    # A= c |00>(<+ -|+<- +|), with c=2J in the manuscript convention.
    a_template = directed_transition_witness_template(
        target_pattern=(0, 0),
        source_patterns=((1, -1), (-1, 1)),
        amplitudes=(xy_matrix_element, xy_matrix_element),
        metadata={"name": "Ared_r_r+1", "support_sites": 2},
    )
    z_template = hermitianize_local_witness_template(
        a_template,
        metadata={"name": "Zred_r_r+1", "support_sites": 2},
    )

    raw = {
        "Y": y_template.instantiate((0,)),
        "A": a_template.instantiate((0, 1)),
        "Z": z_template.instantiate((0, 1)),
    }
    normalized = {
        name: witness.template.normalized("operator_norm").instantiate(witness.variable_indices)
        for name, witness in raw.items()
    }
    return raw, normalized


def tower_state_for_sector(basis_configs: np.ndarray, *, length: int) -> np.ndarray:
    states, labels = spin_one_xy_scar_tower_states(
        basis_configs=basis_configs,
        length=length,
        normalize=True,
    )
    if states.shape[1] != 1:
        raise RuntimeError(f"expected one tower state in a fixed-M basis, found {labels}")
    return states[:, 0]


def tower_symmetry_sector(basis_configs: np.ndarray, scar: np.ndarray, *, length: int):
    n_raised = (TOTAL_SZ + length) // 2
    momentum_index = 0 if n_raised % 2 == 0 else length // 2

    translation = basis_permutation_from_variable_permutation(
        basis_configs,
        np.roll(np.arange(length), 1),
    )
    sector = cyclic_symmetry_sector_basis(
        translation,
        order=length,
        momentum_index=momentum_index,
        labels={"total_sz": TOTAL_SZ},
    )

    # Reflection r -> -r.  k=0 and k=pi sectors are invariant under reflection.
    reflection = basis_permutation_from_variable_permutation(
        basis_configs,
        (-np.arange(length)) % length,
    )
    reflection_value = complex(np.vdot(scar, permutation_matrix(reflection) @ scar))
    reflection_parity = 1 if reflection_value.real >= 0.0 else -1
    sector = refine_sector_by_involution(
        sector,
        reflection,
        eigenvalue=reflection_parity,
        label="reflection",
    )
    return sector, momentum_index, reflection_parity


def projected_witness_square(witness, basis_configs, sector):
    local_operator = witness.embed(basis_configs)
    q_operator = local_operator.conj().T @ local_operator
    return project_operator_to_sector(q_operator, sector)


def periodic_phase_compatible_model(*, length: int, d_z: float):
    return SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=J1_MATRIX,
        d_z=d_z,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=spin_one_xy_periodic_range_couplings(
            length=length,
            distance=3,
            coefficient=J3_MATRIX,
        ),
    )


RAW_WITNESSES, UNIT_WITNESSES = make_spin1_witnesses()
Y_WITNESS = RAW_WITNESSES["Y"]
A_WITNESS = RAW_WITNESSES["A"]
Z_WITNESS = RAW_WITNESSES["Z"]
Y_UNIT = UNIT_WITNESSES["Y"]
A_UNIT = UNIT_WITNESSES["A"]
Z_UNIT = UNIT_WITNESSES["Z"]

witness_norm_df = pd.DataFrame(
    [
        {
            "witness": name,
            "operator_norm_raw": RAW_WITNESSES[name].template.operator_norm,
            "Q_norm_raw": RAW_WITNESSES[name].template.q_operator_norm,
            "Delta_Q_unit": diagnose_local_channel_spectrum(
                UNIT_WITNESSES[name],
                tolerance=TOL,
            ).dark_channel_gap,
            "Q_rank_unit": diagnose_local_channel_spectrum(
                UNIT_WITNESSES[name],
                tolerance=TOL,
            ).rank,
        }
        for name in ("A", "Z", "Y")
    ]
)
display(witness_norm_df)

## A. Exact tower and the three local witnesses

We verify the exact eigenstate residual and the darkness conditions
$A_R|\mathcal S_n\rangle=Z_R|\mathcal S_n\rangle=Y_R|\mathcal S_n\rangle=0$.

In [ ]:
L_REP = 8
model_rep = periodic_phase_compatible_model(length=L_REP, d_z=0.0)
build_rep = model_rep.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
    on_missing="raise",
)
configs_rep = basis_configs_from_build_result(build_rep)
scar_rep = tower_state_for_sector(configs_rep, length=L_REP)
support_rep = np.flatnonzero(np.abs(scar_rep) > TOL)

stability_rep = diagnose_cage_stability(
    build_rep.kinetic,
    support_rep,
    state=scar_rep,
    tolerance=TOL,
)
eigenpair_rep = diagnose_eigenpair(build_rep.hamiltonian, scar_rep)
witness_evaluations = {
    name: evaluate_local_witness_on_states(
        witness,
        basis_configs=configs_rep,
        states=scar_rep,
    )
    for name, witness in RAW_WITNESSES.items()
}

local_rows = []
for descriptor in model_rep.local_term_descriptors(operator_kind="kinetic", term_kind="bond"):
    local_matrix = model_rep.build_local_term(descriptor, build_rep, builder="optimized")
    local_rows.append(
        {
            "term": descriptor.label,
            "sites": descriptor.support_sites,
            "action_norm": float(np.linalg.norm(local_matrix @ scar_rep)),
        }
    )
local_term_df = pd.DataFrame(local_rows)

boundary_scorecard = pd.DataFrame(
    [
        {
            "L": L_REP,
            "M": TOTAL_SZ,
            "full_sector_dimension": configs_rep.shape[0],
            "support_size": support_rep.size,
            "boundary_rank": stability_rep.boundary_rank,
            "boundary_nullity": stability_rep.boundary_nullity,
            "boundary_singular_gap": stability_rep.interference_gap,
            "boundary_residual": stability_rep.state_boundary_residual,
            "internal_residual": stability_rep.state_internal_eigen_residual,
            "full_eigenpair_residual": eigenpair_rep.residual_norm,
            "A_annihilation_residual": witness_evaluations["A"].annihilation_residual,
            "Z_annihilation_residual": witness_evaluations["Z"].annihilation_residual,
            "Y_annihilation_residual": witness_evaluations["Y"].annihilation_residual,
        }
    ]
)

display(boundary_scorecard)
display(local_term_df)
boundary_scorecard.to_csv(DATA_DIR / "boundary_kernel_scorecard.csv", index=False)
local_term_df.to_csv(DATA_DIR / "local_term_annihilation.csv", index=False)
witness_norm_df.to_csv(DATA_DIR / "local_channel_spectra.csv", index=False)

The finite-size check records the tower support, the one-dimensional boundary kernel, the full eigenstate residual, and the local witness residuals.  The three positive observables are normalized to make their thermal activities directly comparable.

## B. Analytical infinite-temperature reference at $D=0$

For fixed $M=-2$ and $L\to\infty$, $q=M/L\to0$ and $p_0\to1/3$.  The normalized activities approach $1/3$, $1/9$, and $2/9$ for $Q_R^Y$, $Q_R^A$, and $Q_R^Z$, respectively.

In [ ]:
formula_rows = []
for length in range(4, 32, 2):
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    formula_rows.append(exact.to_summary_dict())
formula_df = pd.DataFrame(formula_rows)

# Independent direct traces in the qlinks fixed-M basis for ED-accessible sizes.
direct_rows = []
for length in (4, 6, 8, 10):
    result = SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
    ).build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    evaluations = {
        name: evaluate_local_witness_on_diagonal_ensemble(
            witness,
            basis_configs=configs,
        )
        for name, witness in RAW_WITNESSES.items()
    }
    direct_rows.append(
        {
            "length": length,
            "basis_dimension": configs.shape[0],
            "Y2_direct_trace": evaluations["Y"].expectation,
            "A2_direct_trace": evaluations["A"].expectation,
            "Z2_direct_trace": evaluations["Z"].expectation,
            "A2_direct_normalized": evaluations["A"].normalized_expectation,
            "Z2_direct_normalized": evaluations["Z"].normalized_expectation,
        }
    )
direct_df = pd.DataFrame(direct_rows)
activity_df = formula_df.merge(direct_df, how="left", on="length")
activity_df["Y2_direct_minus_formula"] = activity_df["Y2_direct_trace"] - activity_df["y2_activity"]
activity_df["A2_direct_minus_formula"] = activity_df["A2_direct_trace"] - activity_df["directed_q_activity"]
activity_df["Z2_direct_minus_formula"] = activity_df["Z2_direct_trace"] - activity_df["z2_activity"]

display(activity_df.head(8))
activity_df.to_csv(DATA_DIR / "exact_fixed_M_activities.csv", index=False)

# Thermodynamic asymptotes for the fixed-M sequence TOTAL_SZ=-2, where q=M/L -> 0.
p0_infty = 1.0 / 3.0
y2_infty = p0_infty
a2_infty = 2.0 * abs(J1_MATRIX) ** 2 * p0_infty**2
z2_infty = 4.0 * abs(J1_MATRIX) ** 2 * p0_infty**2
a2_infty_normalized = a2_infty / A_WITNESS.template.q_operator_norm
z2_infty_normalized = z2_infty / Z_WITNESS.template.q_operator_norm

fig, ax = plt.subplots(figsize=(3.35, 2.55))
ax.plot(activity_df["length"], activity_df["y2_activity"], marker="o", label=r"$Q^Y_r$")
ax.plot(
    activity_df["length"],
    activity_df["A2_direct_normalized"],
    marker="s",
    label=r"$Q^A_{r,r+1}/(8J^2)$",
)
ax.plot(
    activity_df["length"],
    activity_df["Z2_direct_normalized"],
    marker="^",
    label=r"$Q^Z_{r,r+1}/(8J^2)$",
)
ax.axhline(y2_infty, linestyle="--", linewidth=0.8)
ax.axhline(a2_infty_normalized, linestyle="--", linewidth=0.8)
ax.axhline(z2_infty_normalized, linestyle="--", linewidth=0.8)
ax.set_xlabel(r"System size $L$")
ax.set_ylabel("Normalized thermal activity")
ax.legend(loc="best", frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
save_spin_figure(fig, "exact_fixed_M_three_witness_activities")
plt.show()


## C. Symmetry-resolved microcanonical ETH test for $H_{XY}+H_3$

We use $J_3/J=0.1$, fixed $M=-2$, and the momentum/inversion sector containing the exact tower.  The primary window is

$$|E-E_{\rm scar}|\le J\sqrt L,$$

with nearby prefactors used only to show finite-size window sensitivity.  The exact tower vector is evaluated directly rather than identified with an arbitrary eigensolver vector inside a degenerate multiplet.

In [ ]:
spectral_rows = []
window_sensitivity_rows = []
scan_cache = {}

for length in SIZES:
    t0 = time.perf_counter()
    n_raised = (TOTAL_SZ + length) // 2

    result_zero = periodic_phase_compatible_model(length=length, d_z=0.0).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    configs = basis_configs_from_build_result(result_zero)
    scar = tower_state_for_sector(configs, length=length)
    sector, momentum_index, reflection_parity = tower_symmetry_sector(
        configs,
        scar,
        length=length,
    )
    scar_sector = project_state_to_sector(scar, sector)
    scar_sector /= np.linalg.norm(scar_sector)

    y_sector = project_operator_to_sector(Y_WITNESS.embed(configs), sector)
    z_sector = project_operator_to_sector(Z_WITNESS.embed(configs), sector)
    qy_sector = projected_witness_square(Y_WITNESS, configs, sector)
    qa_sector = projected_witness_square(A_WITNESS, configs, sector)
    qz_sector = projected_witness_square(Z_WITNESS, configs, sector)

    h0_sector = project_operator_to_sector(result_zero.hamiltonian, sector)
    e0, v0 = la.eigh(h0_sector)
    y0 = eigenstate_expectations(qy_sector, v0)
    a0 = eigenstate_expectations(qa_sector, v0)
    z0 = eigenstate_expectations(qz_sector, v0)
    ymean0 = eigenstate_expectations(y_sector, v0)
    zmean0 = eigenstate_expectations(z_sector, v0)
    scar_overlap0 = np.abs(v0.conj().T @ scar_sector) ** 2
    scar_level0 = int(np.argmax(scar_overlap0))
    scar_degenerate_mask0 = np.abs(e0) <= 1.0e-8
    scar_degenerate_weight0 = float(np.sum(scar_overlap0[scar_degenerate_mask0]))

    # The eigensolver basis inside an exactly degenerate E=0 manifold is arbitrary.
    # Evaluate the known tower vector itself rather than calling the maximum-overlap
    # numerical eigenvector "the scar". This is essential for the ETH scatter plot.
    exact_scar_QY = float(np.vdot(scar_sector, qy_sector @ scar_sector).real)
    exact_scar_QA = float(np.vdot(scar_sector, qa_sector @ scar_sector).real / A_WITNESS.template.q_operator_norm)
    exact_scar_QZ = float(np.vdot(scar_sector, qz_sector @ scar_sector).real / Z_WITNESS.template.q_operator_norm)
    if max(abs(exact_scar_QY), abs(exact_scar_QA), abs(exact_scar_QZ)) > 1.0e-9:
        raise RuntimeError(
            f"exact tower is not dark at L={length}: "
            f"QY={exact_scar_QY:.3e}, QA={exact_scar_QA:.3e}, QZ={exact_scar_QZ:.3e}"
        )

    windows0 = {}
    for prefactor in WINDOW_PREFACTORS:
        plan = thermodynamic_energy_window_plan(
            volume=length,
            energy_density=0.0,
            width_prefactor=prefactor,
            local_energy_scale=J_DRAFT,
        )
        window = select_microcanonical_window_by_width(
            e0,
            target_energy=plan.target_energy,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        indices = np.asarray(window.indices, dtype=np.int64)
        windows0[prefactor] = (plan, window, indices)
        window_sensitivity_rows.append(
            {
                "case": "D0",
                "L": length,
                "window_prefactor": prefactor,
                "target_energy": plan.target_energy,
                "requested_half_width": plan.half_width,
                "energy_density_half_width": plan.energy_density_half_width,
                "n_states": window.n_states,
                "center_offset": window.center_offset,
                "tau_Y": float(np.mean(y0[indices])),
                "tau_A_normalized": float(np.mean(a0[indices]) / A_WITNESS.template.q_operator_norm),
                "tau_Z_normalized": float(np.mean(z0[indices]) / Z_WITNESS.template.q_operator_norm),
                "mean_Y": float(np.mean(ymean0[indices])),
                "mean_Z_normalized": float(np.mean(zmean0[indices]) / np.sqrt(Z_WITNESS.template.q_operator_norm)),
            }
        )
    plan0, window0, idx0 = windows0[PRIMARY_WINDOW_PREFACTOR]
    smooth0 = gaussian_spectral_filter(
        e0,
        target_energy=0.0,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(length),
    )
    smooth_weights0 = np.asarray(smooth0.weights, dtype=np.float64)

    # Finite D uses the same subextensive-width rule, centered at E_scar=D L.
    result_d = periodic_phase_compatible_model(length=length, d_z=D_THERMAL).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    np.testing.assert_array_equal(result_d.basis.states, result_zero.basis.states)
    hd_sector = project_operator_to_sector(result_d.hamiltonian, sector)
    ed, vd = la.eigh(hd_sector)
    yd = eigenstate_expectations(qy_sector, vd)
    ad = eigenstate_expectations(qa_sector, vd)
    zd = eigenstate_expectations(qz_sector, vd)
    scar_energy = D_THERMAL * length
    scar_overlap_d = np.abs(vd.conj().T @ scar_sector) ** 2
    scar_level_d = int(np.argmax(scar_overlap_d))
    plan_d = thermodynamic_energy_window_plan(
        volume=length,
        energy_density=D_THERMAL,
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=J_DRAFT,
    )
    windowd = select_microcanonical_window_by_width(
        ed,
        target_energy=scar_energy,
        half_width=plan_d.half_width,
        degeneracy_tolerance=TOL,
    )
    idxd = np.asarray(windowd.indices, dtype=np.int64)
    smoothd = gaussian_spectral_filter(
        ed,
        target_energy=scar_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(length),
    )
    smooth_weights_d = np.asarray(smoothd.weights, dtype=np.float64)

    gap = adjacent_gap_ratio_report(
        e0,
        trim_fraction=0.10,
        degeneracy_tolerance=1.0e-8,
    )
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    residual_d = diagnose_eigenpair(result_d.hamiltonian, scar)

    spectral_rows.append(
        {
            "L": length,
            "M": TOTAL_SZ,
            "n_raised": n_raised,
            "full_M_sector_dimension": configs.shape[0],
            "momentum_index": momentum_index,
            "momentum_over_pi": 2.0 * momentum_index / length,
            "reflection_parity": reflection_parity,
            "resolved_sector_dimension": sector.sector_dimension,
            "J3_over_J": J3_OVER_J,
            "D0_scar_max_single_vector_overlap": scar_overlap0[scar_level0],
            "D0_scar_degenerate_subspace_weight": scar_degenerate_weight0,
            "D0_scar_degenerate_level_count": int(np.sum(scar_degenerate_mask0)),
            "D0_scar_level_energy": e0[scar_level0],
            "D0_exact_scar_QY": exact_scar_QY,
            "D0_exact_scar_QA_normalized": exact_scar_QA,
            "D0_exact_scar_QZ_normalized": exact_scar_QZ,
            "D0_window_requested_half_width": plan0.half_width,
            "D0_window_energy_density_half_width": plan0.energy_density_half_width,
            "D0_window_actual_half_width": window0.half_width,
            "D0_window_state_count": window0.n_states,
            "D0_window_center_offset": window0.center_offset,
            "D0_microcanonical_Y2": float(np.mean(y0[idx0])),
            "D0_microcanonical_A2_normalized": float(np.mean(a0[idx0]) / A_WITNESS.template.q_operator_norm),
            "D0_microcanonical_Z2_normalized": float(np.mean(z0[idx0]) / Z_WITNESS.template.q_operator_norm),
            "D0_microcanonical_Y_mean": float(np.mean(ymean0[idx0])),
            "D0_microcanonical_Y_variance": float(np.mean(y0[idx0]) - np.mean(ymean0[idx0]) ** 2),
            "D0_microcanonical_Z_mean_normalized": float(np.mean(zmean0[idx0]) / np.sqrt(Z_WITNESS.template.q_operator_norm)),
            "D0_microcanonical_Z_variance_normalized": float((np.mean(z0[idx0]) - np.mean(zmean0[idx0]) ** 2) / Z_WITNESS.template.q_operator_norm),
            "D0_smooth_Y2": float(np.dot(smooth_weights0, y0)),
            "D0_smooth_A2_normalized": float(np.dot(smooth_weights0, a0) / A_WITNESS.template.q_operator_norm),
            "D0_smooth_Z2_normalized": float(np.dot(smooth_weights0, z0) / Z_WITNESS.template.q_operator_norm),
            "D0_smooth_effective_state_count": smooth0.effective_state_count,
            "exact_fixed_M_Y2": exact.y2_activity,
            "exact_fixed_M_A2_normalized": exact.directed_q_activity / A_WITNESS.template.q_operator_norm,
            "exact_fixed_M_Z2_normalized": exact.z2_activity / Z_WITNESS.template.q_operator_norm,
            "finiteD_D": D_THERMAL,
            "finiteD_scar_energy": scar_energy,
            "finiteD_scar_level_energy": ed[scar_level_d],
            "finiteD_scar_overlap": scar_overlap_d[scar_level_d],
            "finiteD_scar_residual": residual_d.residual_norm,
            "finiteD_window_requested_half_width": plan_d.half_width,
            "finiteD_window_energy_density_half_width": plan_d.energy_density_half_width,
            "finiteD_window_actual_half_width": windowd.half_width,
            "finiteD_window_state_count": windowd.n_states,
            "finiteD_window_center_offset": windowd.center_offset,
            "finiteD_microcanonical_Y2": float(np.mean(yd[idxd])),
            "finiteD_microcanonical_A2_normalized": float(np.mean(ad[idxd]) / A_WITNESS.template.q_operator_norm),
            "finiteD_microcanonical_Z2_normalized": float(np.mean(zd[idxd]) / Z_WITNESS.template.q_operator_norm),
            "finiteD_smooth_A2_normalized": float(np.dot(smooth_weights_d, ad) / A_WITNESS.template.q_operator_norm),
            "finiteD_sharp_smooth_A_difference": float(abs(np.mean(ad[idxd]) / A_WITNESS.template.q_operator_norm - np.dot(smooth_weights_d, ad) / A_WITNESS.template.q_operator_norm)),
            "mean_gap_ratio_D0": gap.mean_ratio,
            "gap_ratio_count_D0": len(gap.ratios),
            "runtime_seconds": time.perf_counter() - t0,
        }
    )
    scan_cache[length] = {
        "configs": configs,
        "scar": scar,
        "sector": sector,
        "energies_D0": e0,
        "vectors_D0": v0,
        "Y2_D0": y0,
        "A2_D0": a0,
        "Z2_D0": z0,
        "Y_D0": ymean0,
        "Z_D0": zmean0,
        "scar_level_D0": scar_level0,
        "exact_scar_QY": exact_scar_QY,
        "exact_scar_QA_normalized": exact_scar_QA,
        "exact_scar_QZ_normalized": exact_scar_QZ,
        "window_D0": window0,
        "window_plan_D0": plan0,
        "energies_D": ed,
        "vectors_D": vd,
        "Y2_D": yd,
        "A2_D": ad,
        "Z2_D": zd,
        "scar_level_D": scar_level_d,
        "window_D": windowd,
        "gap_report": gap,
    }

spectral_df = pd.DataFrame(spectral_rows)
window_sensitivity_df = pd.DataFrame(window_sensitivity_rows)
display(spectral_df)
spectral_df.to_csv(DATA_DIR / "symmetry_resolved_spectral_evidence.csv", index=False)
window_sensitivity_df.to_csv(DATA_DIR / "D0_microcanonical_window_sensitivity.csv", index=False)

The energy-density width of the window vanishes as $L^{-1/2}$ and exact degeneracies are retained in full.  The fixed-$M$ trace supplies the analytical target; the symmetry-resolved microcanonical average is the ETH comparison.

### ETH scatter and Fig. 3 panels (a)--(c)

In [ ]:
largest_L = max(SIZES)
largest = scan_cache[largest_L]
energies = largest["energies_D0"]
y_values = largest["Y2_D0"]
a_values = largest["A2_D0"] / A_WITNESS.template.q_operator_norm
z_values = largest["Z2_D0"] / Z_WITNESS.template.q_operator_norm
scar_level = largest["scar_level_D0"]

scatter_df = pd.DataFrame(
    {
        "energy": energies,
        "energy_density": energies / largest_L,
        "QY": y_values,
        "QA_normalized": a_values,
        "QZ_normalized": z_values,
        # This only identifies the arbitrary eigensolver vector with maximum
        # tower overlap inside the degenerate zero-energy manifold.
        "is_max_overlap_vector": np.arange(energies.size) == scar_level,
        "is_microcanonical": np.isin(
            np.arange(energies.size),
            np.asarray(largest["window_D0"].indices, dtype=np.int64),
        ),
    }
)
exact_scar_scatter = pd.DataFrame(
    {
        "energy": [0.0],
        "energy_density": [0.0],
        "QY": [largest["exact_scar_QY"]],
        "QA_normalized": [largest["exact_scar_QA_normalized"]],
        "QZ_normalized": [largest["exact_scar_QZ_normalized"]],
    }
)
scatter_df.to_csv(DATA_DIR / "eth_scatter_Lmax_D0.csv", index=False)
exact_scar_scatter.to_csv(DATA_DIR / "eth_scatter_Lmax_exact_tower.csv", index=False)

central = window_sensitivity_df[np.isclose(window_sensitivity_df["window_prefactor"], PRIMARY_WINDOW_PREFACTOR)].copy()


def plot_figure3a(ax, *, length: int | None = None):
    if length is None:
        length = largest_L
    cached = scan_cache[length]
    if length == largest_L:
        subset = scatter_df
    else:
        window_indices = np.asarray(cached["window_D0"].indices, dtype=np.int64)
        subset = pd.DataFrame(
            {
                "energy_density": cached["energies_D0"] / length,
                "QY": cached["Y2_D0"],
                "QA_normalized": cached["A2_D0"] / A_WITNESS.template.q_operator_norm,
                "QZ_normalized": cached["Z2_D0"] / Z_WITNESS.template.q_operator_norm,
                "is_microcanonical": np.isin(np.arange(cached["energies_D0"].size), window_indices),
            }
        )

    # Shade the actual selected energy window. Degeneracy completion may make
    # this slightly wider than the requested c J sqrt(L) interval.
    half_width_density = float(cached["window_D0"].half_width) / float(length)
    ax.axvspan(
        -half_width_density,
        half_width_density,
        alpha=0.10,
        color="0.5",
        label="microcanonical window",
        zorder=0,
    )
    ax.axvline(0.0, linewidth=0.7, linestyle="--", color="0.45", zorder=1)

    ax.scatter(subset["energy_density"], subset["QY"], s=10, alpha=0.65, label=r"$Q^Y_r$")
    ax.scatter(subset["energy_density"], subset["QA_normalized"], s=10, alpha=0.65, label=r"$Q^A_{r,r+1}/(8J^2)$")
    ax.scatter(subset["energy_density"], subset["QZ_normalized"], s=10, alpha=0.65, label=r"$Q^Z_{r,r+1}/(8J^2)$")

    # Plot the analytically known tower vector itself. A numerical diagonalizer
    # may return arbitrary mixtures inside the degenerate E=0 manifold, whose
    # witness expectation need not vanish even though the exact tower is dark.
    scar_values = (
        cached["exact_scar_QY"],
        cached["exact_scar_QA_normalized"],
        cached["exact_scar_QZ_normalized"],
    )
    ax.scatter([0.0], [scar_values[0]], marker="*", s=80, edgecolors="black", linewidths=0.5, label="exact tower", zorder=5)
    ax.set_xlabel(r"Energy density $e=E/L$")
    ax.set_ylabel("Normalized local activity")
    ax.grid(alpha=0.3)
    return ax


def plot_figure3b(ax):
    by_L = window_sensitivity_df.groupby("L")
    for column, label, asymptote in (
        ("tau_Y", r"$Q^Y_r$", 1.0 / 3.0),
        ("tau_A_normalized", r"$Q^A_{r,r+1}/(8J^2)$", 1.0 / 9.0),
        ("tau_Z_normalized", r"$Q^Z_{r,r+1}/(8J^2)$", 2.0 / 9.0),
    ):
        lows = by_L[column].min().reindex(central["L"]).to_numpy()
        highs = by_L[column].max().reindex(central["L"]).to_numpy()
        values = central[column].to_numpy()
        ax.errorbar(
            central["L"],
            values,
            yerr=np.vstack([values - lows, highs - values]),
            marker="o",
            capsize=3,
            label=label,
        )
        ax.axhline(asymptote, linestyle="--", linewidth=0.8)
    ax.set_xlabel(r"System size $L$")
    ax.set_ylabel("Normalized thermal activity")
    ax.grid(alpha=0.3)
    return ax


def plot_figure3c(ax):
    ax.plot(spectral_df["L"], -spectral_df["D0_microcanonical_Y_mean"], marker="o", label=r"$-\langle Y_r\rangle_{\rm mc}$")
    ax.plot(spectral_df["L"], spectral_df["D0_microcanonical_Y_variance"], marker="s", label=r"${\rm Var}_{\rm mc}(Y_r)$")
    ax.plot(spectral_df["L"], spectral_df["D0_microcanonical_Z_mean_normalized"], marker="^", label=r"$\langle Z\rangle_{\rm mc}/\sqrt{8J^2}$")
    ax.plot(spectral_df["L"], spectral_df["D0_microcanonical_Z_variance_normalized"], marker="v", label=r"${\rm Var}_{\rm mc}(Z)/(8J^2)$")
    ax.set_xlabel(r"System size $L$")
    ax.set_ylabel("Mean or variance")
    ax.grid(alpha=0.3)
    return ax


fig, ax = plt.subplots(figsize=(3.35, 2.5))
plot_figure3a(ax)
ax.legend(loc="best", frameon=False, fontsize=7)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_eth_scatter", aliases=("spin1_xy_D0_eth_scatter",))
plt.show()

fig, ax = plt.subplots(figsize=(3.35, 2.5))
plot_figure3b(ax)
ax.legend(loc="best", frameon=False, fontsize=7)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_microcanonical_convergence", aliases=("spin1_xy_D0_microcanonical_convergence",))
plt.show()

fig, ax = plt.subplots(figsize=(3.35, 2.5))
plot_figure3c(ax)
ax.legend(loc="best", frameon=False, fontsize=7)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_mean_variance", aliases=("spin1_xy_D0_mean_variance",))
plt.show()

display(scatter_df.iloc[max(0, scar_level - 3): scar_level + 4])


In [ ]:
# The draft-oriented figures are saved in DATA_DIR by the preceding cell.
print("saved D=0 ETH figures in", DATA_DIR)

## D. Preserving $J_3/J$ scan and Fig. 3 panel (d)

In [ ]:
PRESERVING_J3_PATH = np.array((0.00, 0.05, 0.10, 0.15, 0.20), dtype=float)
preserving_rows = []
for length in SIZES:
    for j3_ratio in PRESERVING_J3_PATH:
        model = SpinOneXYChainModel(
            length=length,
            boundary_condition="periodic",
            j_xy=J1_MATRIX,
            d_z=0.0,
            total_sz=TOTAL_SZ,
            extra_xy_couplings=spin_one_xy_periodic_range_couplings(
                length=length,
                distance=3,
                coefficient=2.0 * j3_ratio * J_DRAFT,
            ),
        )
        result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
        configs = basis_configs_from_build_result(result)
        scar = tower_state_for_sector(configs, length=length)
        sector, _, _ = tower_symmetry_sector(configs, scar, length=length)
        qy_sector = projected_witness_square(Y_WITNESS, configs, sector)
        qa_sector = projected_witness_square(A_WITNESS, configs, sector)
        qz_sector = projected_witness_square(Z_WITNESS, configs, sector)
        y_sector = project_operator_to_sector(Y_WITNESS.embed(configs), sector)
        z_sector = project_operator_to_sector(Z_WITNESS.embed(configs), sector)
        h_sector = project_operator_to_sector(result.hamiltonian, sector)
        energies, vectors = la.eigh(h_sector)
        plan = thermodynamic_energy_window_plan(
            volume=length,
            energy_density=0.0,
            width_prefactor=PRIMARY_WINDOW_PREFACTOR,
            local_energy_scale=J_DRAFT,
        )
        window = select_microcanonical_window_by_width(
            energies,
            target_energy=0.0,
            half_width=plan.half_width,
            degeneracy_tolerance=TOL,
        )
        moments_y = spectral_observable_moments(y_sector, vectors, squared_operator=qy_sector, indices=window.indices)
        moments_z = spectral_observable_moments(z_sector, vectors, squared_operator=qz_sector, indices=window.indices)
        a_values = eigenstate_expectations(qa_sector, vectors)
        smooth = gaussian_spectral_filter(
            energies,
            target_energy=0.0,
            sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(length),
        )
        preserving_rows.append(
            {
                "L": int(length),
                "J3_over_J": float(j3_ratio),
                "window_state_count": int(window.n_states),
                "window_energy_density_half_width": float(plan.energy_density_half_width),
                "tau_Y": float(moments_y.second_moment),
                "tau_A_normalized": float(np.mean(a_values[np.asarray(window.indices, dtype=np.int64)]) / A_WITNESS.template.q_operator_norm),
                "tau_Z_normalized": float(moments_z.second_moment / Z_WITNESS.template.q_operator_norm),
                "mean_Y": float(moments_y.mean),
                "var_Y": float(moments_y.variance),
                "mean_Z_normalized": float(moments_z.mean / np.sqrt(Z_WITNESS.template.q_operator_norm)),
                "var_Z_normalized": float(moments_z.variance / Z_WITNESS.template.q_operator_norm),
                "smooth_tau_A_normalized": float(np.dot(np.asarray(smooth.weights), a_values) / A_WITNESS.template.q_operator_norm),
                "scar_residual": float(diagnose_eigenpair(result.hamiltonian, scar).residual_norm),
            }
        )

preserving_scan_df = pd.DataFrame(preserving_rows)
preserving_scan_df.to_csv(DATA_DIR / "spin1_xy_preserving_j3_scan.csv", index=False)
display(preserving_scan_df)


def plot_figure3d(ax):
    marker_map = {"tau_Y": "o", "tau_A_normalized": "s", "tau_Z_normalized": "^"}
    label_map = {
        "tau_Y": r"$Q^Y_r$",
        "tau_A_normalized": r"$Q^A_{r,r+1}/(8J^2)$",
        "tau_Z_normalized": r"$Q^Z_{r,r+1}/(8J^2)$",
    }
    for length in sorted(preserving_scan_df["L"].unique()):
        subset = preserving_scan_df[preserving_scan_df["L"] == length]
        for column in ("tau_Y", "tau_A_normalized", "tau_Z_normalized"):
            ax.plot(
                subset["J3_over_J"],
                subset[column],
                marker=marker_map[column],
                linewidth=0.9,
                label=label_map[column] + rf", $L={length}$",
            )
    ax.axvline(J3_OVER_J, linestyle="--", linewidth=0.8, color="0.4")
    ax.set_xlabel(r"Preserving exchange ratio $J_3/J$")
    ax.set_ylabel("Normalized thermal activity")
    ax.grid(alpha=0.3)
    return ax


fig, ax = plt.subplots(figsize=(3.35, 2.55))
plot_figure3d(ax)
ax.legend(loc="best", frameon=False, fontsize=6)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_preserving_deformation_scan")
plt.show()

fig = plt.figure(figsize=(7.0, 6.2))
grid = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.30)
axes = [fig.add_subplot(grid[0,0]), fig.add_subplot(grid[0,1]), fig.add_subplot(grid[1,0]), fig.add_subplot(grid[1,1])]
plot_figure3a(axes[0])
plot_figure3b(axes[1])
plot_figure3c(axes[2])
plot_figure3d(axes[3])
for ax, label in zip(axes, ("(a)", "(b)", "(c)", "(d)")):
    ax.text(0.02, 0.98, label, transform=ax.transAxes, ha="left", va="top")
    ax.legend(loc="best", frameon=False, fontsize=6)
fig.tight_layout()
save_spin_figure(fig, "spin1_xy_figure3_combined")
plt.show()


## E. Preserving versus non-preserving exchange deformations

The staggered tower survives exchange processes satisfying the two-parent phase-cancellation condition.  We contrast these with an explicit same-sublattice exchange that violates the condition and gives a nonzero eigenstate residual.

In [ ]:
L_DEF = 6
phases = (-1.0) ** np.arange(L_DEF)
nearest_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=1,
    coefficient=J1_MATRIX,
)
third_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=3,
    coefficient=J3_MATRIX,
)
compatibility = spin_one_xy_phase_compatibility(
    nearest_pairs + third_pairs,
    phases=phases,
)
compatibility_df = pd.DataFrame(
    [
        {
            "site_i": pair[0],
            "site_j": pair[1],
            "coupling": coupling,
            "phase_condition_residual": residual,
            "absolute_residual": abs(residual),
        }
        for pair, coupling, residual in zip(
            compatibility.pairs,
            compatibility.couplings,
            compatibility.residuals,
            strict=True,
        )
    ]
)
display(compatibility_df)
compatibility_df.to_csv(DATA_DIR / "bondwise_phase_compatibility.csv", index=False)

base_result = periodic_phase_compatible_model(length=L_DEF, d_z=D_THERMAL).build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
base_configs = basis_configs_from_build_result(base_result)
base_scar = tower_state_for_sector(base_configs, length=L_DEF)
violating_result = SpinOneXYChainModel(
    length=L_DEF,
    boundary_condition="periodic",
    j_xy=0.0,
    total_sz=TOTAL_SZ,
    extra_xy_couplings=((0, 2, 1.0),),  # same-sublattice exchange violates Eq. (134)
).build(builder="optimized", basis_solver="dfs", sort_basis=True)
np.testing.assert_array_equal(violating_result.basis.states, base_result.basis.states)

violation_rows = []
for epsilon in np.linspace(0.0, 0.20, 11):
    hamiltonian = base_result.hamiltonian + epsilon * violating_result.hamiltonian
    report = diagnose_eigenpair(hamiltonian, base_scar)
    phase_report = spin_one_xy_phase_compatibility(
        nearest_pairs + third_pairs + ((0, 2, epsilon),),
        phases=phases,
    )
    violation_rows.append(
        {
            "epsilon": epsilon,
            "max_phase_condition_residual": phase_report.max_residual,
            "scar_residual": report.residual_norm,
            "scar_variance": report.variance,
        }
    )
violation_df = pd.DataFrame(violation_rows)
display(violation_df)
violation_df.to_csv(DATA_DIR / "phase_condition_violation.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(violation_df["epsilon"], violation_df["scar_residual"], marker="o")
ax.set_xlabel(r"phase-incompatible coupling $\epsilon$")
ax.set_ylabel(r"$\|(H-E)|S_n\rangle\|$")
ax.grid()
fig.tight_layout()
ax.set_yscale("log")
fig.savefig(DATA_DIR / "phase_condition_violation_residual.pdf")

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(violation_df["epsilon"], violation_df["max_phase_condition_residual"], marker="o")
ax.set_xlabel(r"phase-incompatible coupling $\epsilon$")
ax.set_ylabel("max bondwise phase-condition residual")
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "phase_condition_violation_obstruction.pdf")


## F. Spatially varying $D_r$

A site-dependent single-ion anisotropy preserves the tower because every support configuration has $(S_r^z)^2=1$.  We recompute the microcanonical activity at the corresponding tower energy in a translation-breaking example.

In [ ]:
L_INHOM = 6
rng = np.random.default_rng(13)
sites = np.arange(L_INHOM)

# Arbitrary real exchanges between opposite sublattices satisfy Eq. (134) for eta_r=(-1)^r.
# Random bond strengths break translation and reflection while preserving the tower exactly.
inhom_pairs = []
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=1,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(1.5 + 0.8 * rng.random())))
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=3,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(0.2 + 0.8 * rng.random())))

d_profile = 0.4 + 0.4 * rng.random(L_INHOM)
inhom_phase = spin_one_xy_phase_compatibility(
    tuple(inhom_pairs),
    phases=(-1.0) ** sites,
)
assert inhom_phase.is_compatible

inhom_model = SpinOneXYChainModel(
    length=L_INHOM,
    boundary_condition="periodic",
    j_xy=0.0,
    d_z_by_site=tuple(float(value) for value in d_profile),
    total_sz=TOTAL_SZ,
    extra_xy_couplings=tuple(inhom_pairs),
)
inhom_result = inhom_model.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
inhom_configs = basis_configs_from_build_result(inhom_result)
inhom_scar = tower_state_for_sector(inhom_configs, length=L_INHOM)
inhom_residual = diagnose_eigenpair(inhom_result.hamiltonian, inhom_scar)
inhom_scar_energy = float(np.sum(d_profile))

# Spatial symmetries are deliberately broken, so the fixed-M block is already desymmetrized.
inhom_h = inhom_result.hamiltonian.toarray()
inhom_energies, inhom_vectors = la.eigh(inhom_h)
y_local = Y_WITNESS.embed(inhom_configs)
a_local = A_WITNESS.embed(inhom_configs)
z_local = Z_WITNESS.embed(inhom_configs)
y2_inhom = eigenstate_expectations(y_local.conj().T @ y_local, inhom_vectors)
a2_inhom = eigenstate_expectations(a_local.conj().T @ a_local, inhom_vectors)
z2_inhom = eigenstate_expectations(z_local.conj().T @ z_local, inhom_vectors)
inhom_overlap = np.abs(inhom_vectors.conj().T @ inhom_scar)
inhom_scar_level = int(np.argmax(inhom_overlap))
inhom_window = select_microcanonical_window_by_count(
    inhom_energies,
    target_energy=inhom_scar_energy,
    target_count=80,
    include_boundary_degeneracy=True,
)
inhom_indices = np.asarray(inhom_window.indices, dtype=np.int64)
inhom_gap = adjacent_gap_ratio_report(
    inhom_energies,
    trim_fraction=0.10,
    degeneracy_tolerance=1.0e-8,
)

inhom_df = pd.DataFrame(
    [
        {
            "L": L_INHOM,
            "M": TOTAL_SZ,
            "full_sector_dimension": inhom_configs.shape[0],
            "max_phase_condition_residual": inhom_phase.max_residual,
            "scar_energy_expected": inhom_scar_energy,
            "scar_energy_eigensolver": inhom_energies[inhom_scar_level],
            "scar_overlap": inhom_overlap[inhom_scar_level],
            "scar_residual": inhom_residual.residual_norm,
            "window_half_width": inhom_window.half_width,
            "window_state_count": inhom_window.n_states,
            "window_center_offset": inhom_window.center_offset,
            "microcanonical_Y2": float(np.mean(y2_inhom[inhom_indices])),
            "microcanonical_A2": float(np.mean(a2_inhom[inhom_indices])),
            "microcanonical_unit_A": float(np.mean(a2_inhom[inhom_indices]) / A_WITNESS.template.q_operator_norm),
            "microcanonical_Z2": float(np.mean(z2_inhom[inhom_indices])),
            "mean_gap_ratio": inhom_gap.mean_ratio,
            "gap_ratio_count": len(inhom_gap.ratios),
        }
    ]
)
inhom_profile_df = pd.DataFrame({"site": sites, "D_r": d_profile})
inhom_coupling_df = pd.DataFrame(
    [
        {
            "site_i": site_i,
            "site_j": site_j,
            "matrix_element": coupling,
        }
        for site_i, site_j, coupling in inhom_pairs
    ]
)
display(inhom_profile_df)
display(inhom_coupling_df)
display(inhom_df)
inhom_profile_df.to_csv(DATA_DIR / "inhomogeneous_D_profile.csv", index=False)
inhom_coupling_df.to_csv(DATA_DIR / "inhomogeneous_phase_compatible_couplings.csv", index=False)
inhom_df.to_csv(DATA_DIR / "inhomogeneous_D_evidence.csv", index=False)

## G. Additional finite-size deformation diagnostics

The following tables record which declared perturbation directions preserve the tower and how the local witness and thermal activity behave along representative preserving paths.  These are supporting diagnostics for the deformation discussion; the main ETH evidence is contained in the preceding sections.

In [ ]:
L_STABILITY = 6
base_stability_model = periodic_phase_compatible_model(length=L_STABILITY, d_z=D_THERMAL)
base_stability = base_stability_model.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
stability_configs = basis_configs_from_build_result(base_stability)
stability_scar = tower_state_for_sector(stability_configs, length=L_STABILITY)
stability_support = np.flatnonzero(np.abs(stability_scar) > TOL)


def perturbation_matrix(*, pairs=(), d_profile=None, h_profile=None):
    model = SpinOneXYChainModel(
        length=L_STABILITY,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=tuple(pairs),
        d_z_by_site=None if d_profile is None else tuple(complex(x) for x in d_profile),
        h_z_by_site=None if h_profile is None else tuple(complex(x) for x in h_profile),
    )
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    np.testing.assert_array_equal(result.basis.states, base_stability.basis.states)
    return result.hamiltonian


nearest_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=1,
    coefficient=1.0,
)
third_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=3,
    coefficient=1.0,
)
second_unit = spin_one_xy_periodic_range_couplings(
    length=L_STABILITY,
    distance=2,
    coefficient=1.0,
)

def one_hot(site):
    return tuple(1.0 if index == site else 0.0 for index in range(L_STABILITY))

alphabets = {
    "odd_range_real": [
        perturbation_matrix(pairs=((i, j, coefficient),))
        for i, j, coefficient in (*nearest_unit, *third_unit)
    ],
    "odd_range_imaginary": [
        perturbation_matrix(pairs=((i, j, 1.0j),))
        for i, j, _coefficient in (*nearest_unit, *third_unit)
    ],
    "inhomogeneous_Dr": [
        perturbation_matrix(d_profile=one_hot(site))
        for site in range(L_STABILITY)
    ],
    "inhomogeneous_hr": [
        perturbation_matrix(h_profile=one_hot(site))
        for site in range(L_STABILITY)
    ],
    "even_range_real": [
        perturbation_matrix(pairs=((i, j, coefficient),))
        for i, j, coefficient in second_unit
    ],
}

obstruction_rows = []
obstruction_spectra = []
for alphabet_name, perturbations in alphabets.items():
    hierarchy = cage_compatibility_hierarchy_from_hamiltonians(
        base_stability.hamiltonian,
        perturbations,
        stability_support,
        stability_scar,
        coefficient_field="real",
        tolerance=TOL,
    )
    first_order = hierarchy.first_order
    obstruction_rows.append(
        {
            "alphabet": alphabet_name,
            "n_parameters": first_order.n_parameters,
            "obstruction_rank": first_order.rank,
            "first_order_compatible_dimension": first_order.compatible_dimension,
            "fixed_state_compatible_dimension": hierarchy.fixed_state.compatible_dimension,
            "tangent_only_dimension": hierarchy.tangent_only_dimension,
        }
    )
    for index, value in enumerate(first_order.singular_values):
        obstruction_spectra.append(
            {
                "alphabet": alphabet_name,
                "singular_index": index,
                "singular_value": float(value),
            }
        )

obstruction_df = pd.DataFrame(obstruction_rows)
obstruction_spectrum_df = pd.DataFrame(obstruction_spectra)
cage_conditioning = cage_jacobian_conditioning_from_hamiltonian(
    base_stability.hamiltonian,
    stability_support,
    stability_scar,
    tolerance=TOL,
)

display(obstruction_df)
display(pd.DataFrame([cage_conditioning.to_summary_dict()]))
obstruction_df.to_csv(DATA_DIR / "deformation_obstruction_scorecard.csv", index=False)
obstruction_spectrum_df.to_csv(DATA_DIR / "deformation_obstruction_spectra.csv", index=False)
pd.DataFrame([cage_conditioning.to_summary_dict()]).to_csv(
    DATA_DIR / "cage_jacobian_conditioning.csv",
    index=False,
)

# Draft-oriented deformation figures.
scorecard_plot_df = obstruction_df.copy()
scorecard_plot_df = scorecard_plot_df.sort_values(
    ["first_order_compatible_dimension", "n_parameters"],
    ascending=[False, True],
).reset_index(drop=True)
scorecard_plot_df["obstructed_dimension"] = (
    scorecard_plot_df["n_parameters"] - scorecard_plot_df["first_order_compatible_dimension"]
)
scorecard_plot_df["floating_compatible_dimension"] = scorecard_plot_df["tangent_only_dimension"]
labels = scorecard_plot_df["alphabet"].tolist()
ypos = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.barh(ypos, scorecard_plot_df["first_order_compatible_dimension"], label="first-order compatible")
ax.barh(
    ypos,
    scorecard_plot_df["obstructed_dimension"],
    left=scorecard_plot_df["first_order_compatible_dimension"],
    label="obstructed",
)
ax.plot(
    scorecard_plot_df["fixed_state_compatible_dimension"],
    ypos,
    marker="o",
    linestyle="None",
    label="fixed-state compatible",
)
ax.set_yticks(ypos, labels)
ax.set_xlabel("parameter-space dimension")
ax.set_ylabel("deformation alphabet")
ax.legend(loc="best", fontsize=9)
ax.grid(axis="x")
fig.tight_layout()
fig.savefig(DATA_DIR / "deformation_obstruction_scorecard.pdf")

fig, ax = plt.subplots(figsize=(6.8, 4.0))
for alphabet, frame in obstruction_spectrum_df.groupby("alphabet", sort=False):
    ordered = frame.sort_values("singular_index")
    ax.semilogy(
        ordered["singular_index"] + 1,
        np.maximum(ordered["singular_value"], 1.0e-16),
        marker="o",
        label=alphabet,
    )
ax.set_xlabel("singular-value index")
ax.set_ylabel("first-order obstruction singular value")
ax.legend(loc="best", fontsize=8)
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "deformation_obstruction_singular_spectra.pdf")


### Local witness gap

The positive operator $Q_R^A=A_R^\dagger A_R$ has a finite nonzero local eigenvalue after normalization.  We record this local gap separately from the full many-body cage residual.

In [ ]:
a_local_unit = np.asarray(A_UNIT.local_operator, dtype=np.complex128)
local_dark_vector = np.asarray([0.0, 1.0, -1.0], dtype=np.complex128) / np.sqrt(2.0)
scale_perturbation = a_local_unit.copy()
imbalance_perturbation = np.zeros_like(a_local_unit)
imbalance_perturbation[0, 1] = 1.0
imbalance_perturbation[0, 2] = -1.0

local_obstruction = linearized_cage_obstruction(
    a_local_unit,
    local_dark_vector,
    (scale_perturbation, imbalance_perturbation),
    coefficient_field="real",
    tolerance=TOL,
)
local_q_spectrum = diagnose_local_channel_spectrum(A_UNIT, tolerance=TOL)
local_channel_df = pd.DataFrame(
    [
        {
            "perturbation": "common_scale",
            "fixed_dark_vector_residual": float(
                np.linalg.norm(scale_perturbation @ local_dark_vector)
            ),
            "first_order_obstruction_residual": float(
                np.linalg.norm(local_obstruction.obstruction_matrix[:, 0])
            ),
        },
        {
            "perturbation": "source_imbalance",
            "fixed_dark_vector_residual": float(
                np.linalg.norm(imbalance_perturbation @ local_dark_vector)
            ),
            "first_order_obstruction_residual": float(
                np.linalg.norm(local_obstruction.obstruction_matrix[:, 1])
            ),
        },
    ]
)
local_channel_summary_df = pd.DataFrame(
    [
        {
            "obstruction_rank": local_obstruction.rank,
            "compatible_dimension": local_obstruction.compatible_dimension,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "Q_rank": local_q_spectrum.rank,
            "Q_nullity": local_q_spectrum.nullity,
            "witness_radius_bonds": 1,
        }
    ]
)
display(local_channel_df)
display(local_channel_summary_df)
local_channel_df.to_csv(DATA_DIR / "directed_local_channel_perturbations.csv", index=False)
local_channel_summary_df.to_csv(
    DATA_DIR / "directed_local_channel_stability.csv",
    index=False,
)

### Uniform finite-$D$ path

For each $D$, the microcanonical window is recentered at the exact tower energy $E_{\rm scar}=DL$.

In [ ]:
D_PATH = D_THERMAL + np.linspace(-0.20, 0.20, 5)
finite_d_margin_rows = []
for d_value in D_PATH:
    model = periodic_phase_compatible_model(length=L_STABILITY, d_z=float(d_value))
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    scar = tower_state_for_sector(configs, length=L_STABILITY)
    sector, _, _ = tower_symmetry_sector(configs, scar, length=L_STABILITY)
    projected_h = project_operator_to_sector(result.hamiltonian, sector)
    energies, vectors = la.eigh(projected_h)
    projected_q = projected_witness_square(A_UNIT, configs, sector)
    activities = eigenstate_expectations(projected_q, vectors)
    plan = thermodynamic_energy_window_plan(
        volume=L_STABILITY,
        energy_density=float(d_value),
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=J_DRAFT,
    )
    window = select_microcanonical_window_by_width(
        energies,
        target_energy=plan.target_energy,
        half_width=plan.half_width,
        degeneracy_tolerance=TOL,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    smooth = gaussian_spectral_filter(
        energies,
        target_energy=plan.target_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(L_STABILITY),
    )
    smooth_activity = float(np.dot(np.asarray(smooth.weights), activities))
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        result.hamiltonian,
        np.flatnonzero(np.abs(scar) > TOL),
        scar,
        tolerance=TOL,
    )
    finite_d_margin_rows.append(
        {
            "D": float(d_value),
            "coupling_path_parameter": float(np.sqrt(L_STABILITY) * (d_value - D_THERMAL)),
            "scar_residual": diagnose_eigenpair(result.hamiltonian, scar).residual_norm,
            "Delta_cage": conditioning.cage_gap,
            "window_state_count": window.n_states,
            "window_requested_half_width": plan.half_width,
            "window_actual_half_width": window.half_width,
            "window_energy_density_half_width": plan.energy_density_half_width,
            "sharp_microcanonical_activity": float(np.mean(activities[indices])),
            "smooth_filtered_activity": smooth_activity,
            "sharp_smooth_difference": float(abs(np.mean(activities[indices]) - smooth_activity)),
            "smooth_effective_state_count": smooth.effective_state_count,
        }
    )
finite_d_margin_df = pd.DataFrame(finite_d_margin_rows)
finite_d_margin = thermal_activity_margin_from_samples(
    finite_d_margin_df["coupling_path_parameter"],
    finite_d_margin_df["smooth_filtered_activity"],
    reference_parameter=0.0,
    tolerance=TOL,
)
display(finite_d_margin_df)
display(pd.DataFrame([finite_d_margin.to_summary_dict()]))
finite_d_margin_df.to_csv(DATA_DIR / "finite_D_directed_thermal_path.csv", index=False)
pd.DataFrame([finite_d_margin.to_summary_dict()]).to_csv(
    DATA_DIR / "finite_D_directed_thermal_margin.csv",
    index=False,
)

### Inhomogeneous-$D_r$ path

The bond background is kept phase compatible while $D_r$ changes along one normalized direction.  The exact tower residual and the energy-matched directed-witness activity are evaluated at every point.

In [ ]:
L_MARGIN_INHOM = 6
rng_margin = np.random.default_rng(23)
inhom_margin_pairs = []
for distance, offset, width in ((1, 1.2, 0.7), (3, 0.2, 0.6)):
    for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
        length=L_MARGIN_INHOM,
        distance=distance,
        coefficient=1.0,
    ):
        inhom_margin_pairs.append(
            (site_i, site_j, float(offset + width * rng_margin.random()))
        )
base_d_profile = 0.35 + 0.45 * rng_margin.random(L_MARGIN_INHOM)
d_direction = rng_margin.normal(size=L_MARGIN_INHOM)
d_direction /= np.linalg.norm(d_direction)
G_PATH = np.linspace(-0.20, 0.20, 5)
inhom_margin_rows = []
inhom_base_conditioning = None
for g_value in G_PATH:
    profile = base_d_profile + float(g_value) * d_direction
    model = SpinOneXYChainModel(
        length=L_MARGIN_INHOM,
        boundary_condition="periodic",
        j_xy=0.0,
        d_z_by_site=tuple(float(value) for value in profile),
        total_sz=TOTAL_SZ,
        extra_xy_couplings=tuple(inhom_margin_pairs),
    )
    result = model.build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    scar = tower_state_for_sector(configs, length=L_MARGIN_INHOM)
    energies, vectors = la.eigh(result.hamiltonian.toarray())
    q_local = A_UNIT.embed(configs)
    q_operator = q_local.conj().T @ q_local
    activities = eigenstate_expectations(q_operator, vectors)
    scar_energy = float(np.sum(profile))
    plan = thermodynamic_energy_window_plan(
        volume=L_MARGIN_INHOM,
        energy_density=scar_energy / L_MARGIN_INHOM,
        width_prefactor=PRIMARY_WINDOW_PREFACTOR,
        local_energy_scale=J_DRAFT,
    )
    window = select_microcanonical_window_by_width(
        energies,
        target_energy=scar_energy,
        half_width=plan.half_width,
        degeneracy_tolerance=TOL,
    )
    indices = np.asarray(window.indices, dtype=np.int64)
    smooth = gaussian_spectral_filter(
        energies,
        target_energy=scar_energy,
        sigma=SMOOTH_SIGMA_PREFACTOR * J_DRAFT * np.sqrt(L_MARGIN_INHOM),
    )
    smooth_activity = float(np.dot(np.asarray(smooth.weights), activities))
    conditioning = cage_jacobian_conditioning_from_hamiltonian(
        result.hamiltonian,
        np.flatnonzero(np.abs(scar) > TOL),
        scar,
        tolerance=TOL,
    )
    if abs(float(g_value)) <= TOL:
        inhom_base_conditioning = conditioning
    inhom_margin_rows.append(
        {
            "g": float(g_value),
            "scar_energy": scar_energy,
            "scar_energy_density": scar_energy / L_MARGIN_INHOM,
            "scar_residual": diagnose_eigenpair(result.hamiltonian, scar).residual_norm,
            "Delta_cage": conditioning.cage_gap,
            "window_state_count": window.n_states,
            "window_requested_half_width": plan.half_width,
            "window_actual_half_width": window.half_width,
            "window_energy_density_half_width": plan.energy_density_half_width,
            "sharp_microcanonical_activity": float(np.mean(activities[indices])),
            "smooth_filtered_activity": smooth_activity,
            "sharp_smooth_difference": float(abs(np.mean(activities[indices]) - smooth_activity)),
            "smooth_effective_state_count": smooth.effective_state_count,
        }
    )
if inhom_base_conditioning is None:
    raise RuntimeError("the inhomogeneous path must include g=0")
inhom_margin_df = pd.DataFrame(inhom_margin_rows)
inhom_margin = thermal_activity_margin_from_samples(
    inhom_margin_df["g"],
    inhom_margin_df["smooth_filtered_activity"],
    reference_parameter=0.0,
    tolerance=TOL,
)
display(inhom_margin_df)
display(pd.DataFrame([inhom_margin.to_summary_dict()]))
inhom_margin_df.to_csv(DATA_DIR / "inhomogeneous_D_directed_thermal_path.csv", index=False)
pd.DataFrame([inhom_margin.to_summary_dict()]).to_csv(
    DATA_DIR / "inhomogeneous_D_directed_thermal_margin.csv",
    index=False,
)

stability_profile_df = pd.DataFrame(
    [
        {
            "case": "uniform_finite_D",
            "L": L_STABILITY,
            "Delta_cage": float(finite_d_margin_df.loc[np.argmin(np.abs(finite_d_margin_df["coupling_path_parameter"])), "Delta_cage"]),
            "witness_radius": 1,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "tau_Q_smooth": finite_d_margin.reference_activity,
            "chi_Q_smooth": finite_d_margin.susceptibility_bound,
            "half_activity_radius": finite_d_margin.half_activity_radius,
            "max_sharp_smooth_difference": finite_d_margin_df["sharp_smooth_difference"].max(),
        },
        {
            "case": "inhomogeneous_Dr",
            "L": L_MARGIN_INHOM,
            "Delta_cage": inhom_base_conditioning.cage_gap,
            "witness_radius": 1,
            "Delta_Q": local_q_spectrum.dark_channel_gap,
            "tau_Q_smooth": inhom_margin.reference_activity,
            "chi_Q_smooth": inhom_margin.susceptibility_bound,
            "half_activity_radius": inhom_margin.half_activity_radius,
            "max_sharp_smooth_difference": inhom_margin_df["sharp_smooth_difference"].max(),
        },
    ]
)
display(stability_profile_df)
stability_profile_df.to_csv(DATA_DIR / "predictive_stability_profile.csv", index=False)

### Supporting deformation figures

These plots are optional manuscript or appendix material.  They show the energy-matched directed-witness activity and the finite-size cage-conditioning scale along the two preserving paths.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(finite_d_margin_df["coupling_path_parameter"], finite_d_margin_df["sharp_microcanonical_activity"], marker="o", label="sharp microcanonical")
ax.plot(finite_d_margin_df["coupling_path_parameter"], finite_d_margin_df["smooth_filtered_activity"], marker="s", label="smooth filter")
ax.axhline(finite_d_margin.reference_activity, linestyle="--", label=r"$\widetilde\tau_Q(0)$")
ax.set_xlabel(r"uniform-$D$ coupling distance $g$")
ax.set_ylabel(r"normalized $\langle A^\dagger A\rangle$")
ax.legend(loc="best")
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "finite_D_directed_thermal_margin_curve.pdf")

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(inhom_margin_df["g"], inhom_margin_df["sharp_microcanonical_activity"], marker="o", label="sharp microcanonical")
ax.plot(inhom_margin_df["g"], inhom_margin_df["smooth_filtered_activity"], marker="s", label="smooth filter")
ax.axhline(inhom_margin.reference_activity, linestyle="--", label=r"$\widetilde\tau_Q(0)$")
ax.set_xlabel(r"inhomogeneous-$D_r$ path parameter $g$")
ax.set_ylabel(r"normalized $\langle A^\dagger A\rangle$")
ax.legend(loc="best")
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "inhomogeneous_D_directed_thermal_margin_curve.pdf")

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(finite_d_margin_df["coupling_path_parameter"], finite_d_margin_df["Delta_cage"], marker="o", label="uniform $D$")
ax.plot(inhom_margin_df["g"], inhom_margin_df["Delta_cage"], marker="s", label="inhomogeneous $D_r$")
ax.set_xlabel(r"deformation path parameter")
ax.set_ylabel(r"cage-conditioning gap $\Delta_{\rm cage}$")
ax.legend(loc="best")
ax.grid()
fig.tight_layout()
fig.savefig(DATA_DIR / "spin1_xy_cage_conditioning_paths.pdf")

summary_plot_df = stability_profile_df.set_index("case")
for quantity, filename in [
    ("Delta_cage", "predictive_stability_delta_cage.pdf"),
    ("Delta_Q", "predictive_stability_delta_Q.pdf"),
    ("tau_Q_smooth", "predictive_stability_tau_Q.pdf"),
    ("chi_Q_smooth", "predictive_stability_chi_Q.pdf"),
]:
    fig, ax = plt.subplots(figsize=(5.8, 3.6))
    ax.bar(summary_plot_df.index.tolist(), summary_plot_df[quantity].to_numpy())
    ax.set_ylabel(quantity)
    ax.set_xlabel("benchmark case")
    ax.grid(axis="y")
    fig.tight_layout()
    fig.savefig(DATA_DIR / filename)

## Output manifest

In [ ]:
manifest = pd.DataFrame(
    [
        {"file": path.name, "bytes": path.stat().st_size}
        for path in sorted(DATA_DIR.glob("*.csv"))
    ]
)
display(manifest)
manifest.to_csv(DATA_DIR / "manifest.csv", index=False)
print("All numerical tables were written to", DATA_DIR)

## Numerical evidence represented in this notebook

- $H_{XY}+H_3$ microcanonical ETH scatter and convergence for all three witnesses.
- Exact zero activity of the staggered tower, evaluated with the analytical tower vector even when the cage energy is degenerate.
- Preserving $J_3/J$ scan for the missing Fig. 3 deformation panel.
- Explicit preserving/non-preserving exchange tests.
- Energy-matched finite-$D$ and inhomogeneous-$D_r$ checks.